# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Inspecting the tables on Hugging face for the most suitable one for my lane

In [ ]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [ ]:
from datasets import load_dataset_builder           # Import the function for inspecting a dataset's structure

# Load the dataset builder for the specified FlyRank warehouse table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",         # Hugging Face dataset repository
    "fact_content_daily_performance",       # Table to inspect
    token=HF_TOKEN                          # Hugging Face access token for authentication
)

# Display the dataset's features (column names and their data types)
print(builder.info.features)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

In [ ]:
# Load the dataset builder for the dim_content table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)
print(builder.info.features)

{'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'keyword_hash_id': Value('string'), 'url_hash_id': Value('string'), 'keyword_char_count': Value('int64'), 'keyword_token_count': Value('int64'), 'url_char_count': Value('int64'), 'content_created_date': Value('date32'), 'content_updated_date': Value('date32'), 'content_type': Value('string'), 'search_volume': Value('int64'), 'competition': Value('float64'), 'competition_level': Value('string'), 'cpc': Value('float64'), 'main_intent': Value('string'), 'backlinks': Value('int64'), 'category_count': Value('int64'), 'keyword_created_date': Value('date32'), 'provider_used': Value('string'), 'model_used': Value('string'), 'char_count': Value('int64'), 'word_count': Value('int64'), 'last_optimized_date': Value('date32'), 'optimization_eligible_date': Value('date32'), 'is_published': Value('bool'), 'is_deleted': Value('bool')}


In [ ]:
# Load the dataset for the fact_content_query_90d table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

print(builder.info.features)

{'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'query_hash_id': Value('string'), 'query_char_count': Value('int64'), 'query_token_count': Value('int64'), 'window_start': Value('date32'), 'window_end': Value('date32'), 'impressions_90d': Value('int64'), 'clicks_90d': Value('int64'), 'impressions_last30': Value('int64'), 'clicks_last30': Value('int64'), 'impressions_prev30': Value('int64'), 'clicks_prev30': Value('int64'), 'avg_position_90d': Value('float64'), 'avg_position_last30': Value('float64'), 'avg_position_prev30': Value('float64'), 'content_total_impressions_90d': Value('int64'), 'content_visible_query_count': Value('int64'), 'rare_query_count': Value('int64'), 'rare_impressions_share': Value('float64'), 'anonymized_impressions_share': Value('float64')}


fact_content_daily_performance and dim_content tables contain useful signals for my lane: Refresh / Content Opportunity Scoring

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one content item for one client.

For this analysis, I am using data from March 1, 2026 to March 31, 2026

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

### Five features and availability at the decision moment

| Feature                | Knowable at the decision moment because…                                                                      |
| ---------------------- | ------------------------------------------------------------------------------------------------------------- |
| `gsc_impressions`      | The March 2026 search impressions have already been recorded by the decision moment after March 31.           |
| `gsc_clicks`           | The March 2026 search clicks have already been recorded by the decision moment after March 31.                |
| `gsc_avg_position`     | The March 2026 search-position observations have already been recorded by the decision moment after March 31. |
| `ga4_pageviews`        | The March 2026 pageviews have already been recorded by the decision moment after March 31.                    |
| `ga4_engaged_sessions` | The March 2026 engaged sessions have already been recorded by the decision moment after March 31.             |


### Label / Proxy

* `is_declining_label` - a proxy for whether a content item's performance is declining and may warrant review or refresh.

### Context

* `client_hash_id`
* `content_hash_id`
* `report_date`
* `gsc_data_available`
* `ga4_data_available`

### Excluded

* `ga4_total_engagement_sec` — excluded because it overlaps with the engagement information represented by `ga4_pageviews` and `ga4_engaged_sessions`.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)


In [ ]:
# Define the Hugging Face dataset path
rel = "hf://datasets/FlyRank/internship-warehouse"

# Execute a SQL query in DuckDB to count all rows
# in the fact_content_daily_performance table
con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [ ]:
# Query the fact_content_daily_performance table using DuckDB,
# automatically read partition values from the file paths
# and return only the first 5 rows
con.sql(
    f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    LIMIT 5
    """
)


┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

Grain Verification

In [ ]:
# Verify the proposed grain of the daily performance table.
# Expected grain: one row per report_date × client × content.

grain_check = con.sql(
    f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS c
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use March 2026 as the mid-panel development month.
    WHERE month = '2026-03'

    -- Group by the proposed unit of analysis.
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id

    -- Return any combinations that occur more than once.
    HAVING COUNT(*) > 1

    -- We only need to see whether duplicates exist.
    LIMIT 5
    """
)

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

Counts and windows Verification

In [ ]:
# Verify the row count and date span of the March 2026 analysis slice.

march_summary = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use the mid-panel month selected for development.
    WHERE month = '2026-03'
    """
)

march_summary

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

Availability Check

In [ ]:
# Check how many March 2026 rows have both GSC and GA4 data available.

availability_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS rows_with_both_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    """
)

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┐
│ rows_with_both_available │
│          int64           │
├──────────────────────────┤
│                   364347 │
└──────────────────────────┘

Missing Values Check

In [ ]:
# Check the percentage of NULL values in the five selected features
# for the March 2026 slice.
#
# Each AVG(CASE...) calculates the proportion of rows where
# that particular field is NULL.

con.sql(
    f"""
    SELECT
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END)
            AS gsc_impressions_missing,

        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END)
            AS gsc_clicks_missing,

        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END)
            AS gsc_avg_position_missing,

        AVG(CASE WHEN ga4_pageviews IS NULL THEN 1.0 ELSE 0 END)
            AS ga4_pageviews_missing,

        AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END)
            AS ga4_engaged_sessions_missing

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    WHERE month = '2026-03'
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────────────┬──────────────────────────┬───────────────────────┬──────────────────────────────┐
│ gsc_impressions_missing │ gsc_clicks_missing │ gsc_avg_position_missing │ ga4_pageviews_missing │ ga4_engaged_sessions_missing │
│         double          │       double       │          double          │        double         │            double            │
├─────────────────────────┼────────────────────┼──────────────────────────┼───────────────────────┼──────────────────────────────┤
│                     0.0 │                0.0 │       0.6330736407035681 │   0.30673966592889734 │          0.30673966592889734 │
└─────────────────────────┴────────────────────┴──────────────────────────┴───────────────────────┴──────────────────────────────┘

Five Feature Frame

In [ ]:
# Build the March 2026 feature frame at the original daily grain.
# Each row remains one content item for one client on one day.
#
# The five selected performance fields are kept as features.
# The IDs and date are retained only to identify each observation
# and preserve the time dimension.

march_features = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        -- Search performance features
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        -- Engagement features
        ga4_pageviews,
        ga4_engaged_sessions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    WHERE month = '2026-03'

      -- Keep only rows where both data sources are available.
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    """
).df()

march_features

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_65de48885f4ef01b,content_5c80451459c29b4a,2026-03-01,5,0,5.400000,1,0
1,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2026-03-01,39,0,5.666667,2,0
2,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03-01,179,0,5.156425,2,0
3,client_65de48885f4ef01b,content_6b0149a80607dac3,2026-03-01,72,0,7.694444,1,0
4,client_65de48885f4ef01b,content_62673eea26c31c17,2026-03-01,3282,1,6.167885,1,0
...,...,...,...,...,...,...,...,...
364342,client_20259bd6705d81d4,content_daf1fa4c69738a93,2026-03-31,168,3,15.613095,1,0
364343,client_20259bd6705d81d4,content_8ea16f0ba99969aa,2026-03-31,439,0,5.911162,1,0
364344,client_20259bd6705d81d4,content_5b1533eed3a1f178,2026-03-31,156,1,25.064103,1,0
364345,client_20259bd6705d81d4,content_f712a9db831acfd5,2026-03-31,154,1,18.311688,1,0


In [ ]:
# Confirm the date range

con.sql(
    f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    """
)

┌────────────┬────────────┐
│ first_date │ last_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┘

Leakage

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np


# Load the feature/label frame we just created.
# We use the same March warehouse data for this quick leakage demonstration.

df = con.sql(
    f"""
    WITH daily_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_pageviews,
            ga4_engaged_sessions
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ),

    ranked_daily AS (
        SELECT
            *,
            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY gsc_impressions
            ) AS impressions_rank,

            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY gsc_clicks
            ) AS clicks_rank,

            1 - PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY gsc_avg_position
            ) AS position_rank,

            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY ga4_pageviews
            ) AS pageviews_rank,

            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY ga4_engaged_sessions
            ) AS engagement_rank

        FROM daily_data
    ),

    scored_daily AS (
        SELECT
            *,
            (
                impressions_rank
                + clicks_rank
                + position_rank
                + pageviews_rank
                + engagement_rank
            ) / 5 AS performance_score
        FROM ranked_daily
    ),

    trends AS (
        SELECT
            client_hash_id,
            content_hash_id,
            REGR_SLOPE(
                performance_score,
                DATE_DIFF(
                    'day',
                    DATE '2026-03-01',
                    report_date
                )
            ) AS performance_trend
        FROM scored_daily
        GROUP BY
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) >= 2
    ),

    features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS gsc_impressions,
            AVG(gsc_clicks) AS gsc_clicks,
            AVG(gsc_avg_position) AS gsc_avg_position,
            AVG(ga4_pageviews) AS ga4_pageviews,
            AVG(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM daily_data
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.*,
        t.performance_trend,
        t.performance_trend < 0 AS is_declining_label

    FROM features AS f

    INNER JOIN trends AS t
        ON f.client_hash_id = t.client_hash_id
        AND f.content_hash_id = t.content_hash_id
    """
).df()




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# The five features required by the assignment.
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# The proxy label.
y = df["is_declining_label"].astype(int)




In [ ]:
# Precision@50: among the 50 highest-ranked items,
# what proportion are actually labelled as declining?
def precision_at_k(scores, y, k=50):
    top_k = np.argsort(scores)[::-1][:k]
    return y.iloc[top_k].mean()


HONEST MODEL

In [ ]:

# Only the five legitimate features are provided to the model.

X_honest = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

honest = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
).fit(X_honest, y)

honest_score = precision_at_k(
    honest.predict_proba(X_honest)[:, 1],
    y,
    50
)

print(f"Honest tree Precision@50: {honest_score:.3f}")


Honest tree Precision@50: 0.840


LEAKY MODEL

`performance_trend` is deliberately added.

THIS IS THE TRAP:
is_declining_label was directly created from performance_trend.
Therefore performance_trend contains the answer.


In [ ]:

X_leaky = (
    df[features + ["performance_trend"]]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

leaky = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
).fit(X_leaky, y)

leaky_score = precision_at_k(
    leaky.predict_proba(X_leaky)[:, 1],
    y,
    50
)


print(f"Leaky tree Precision@50:  {leaky_score:.3f}")

Leaky tree Precision@50:  1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Unbalanced history:** Clients have different amounts of available search and analytics history, so performance comparisons across the same calendar period may not be equally reliable for every client.

* **GSC-only early rows:** Some clients have Search Console data available before GA4 data becomes available. These rows cannot be treated as complete performance observations because engagement information is unavailable.

* **Window overlap:** The 90-day query data can overlap with the later part of the performance window. Care must therefore be taken to ensure that features use only information that would have been available at the decision moment.

* **No causal explanation:** The data can show that a content item's performance is declining, but it cannot by itself tell us why the decline happened or whether refreshing the content will cause performance to improve.

* **No direct business outcome:** Performance signals such as impressions, clicks, pageviews, and engaged sessions do not by themselves prove that refreshing a page will increase revenue or conversions.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.